# Generate Skewed Data 

## Import Modules

In [ ]:
from pyspark.sql import SparkSession  # noqa: E402
import pyspark.sql.functions as F  # noqa: E402
from pyspark.sql import Window  # noqa: E402
from pyspark.sql.functions import udf  # noqa: E402
from pyspark.sql.types import StringType  # noqa: E402

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import random  # noqa: E402
import warnings  # noqa: E402
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

In [ ]:
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "8g")
    .config("spark.driver.cores", "4")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("ERROR")

# Setting Up Transactions Data

In [ ]:
customers_file = "../../data/data_skew/raw_customers.csv"
txns_file = "../../data/data_skew/raw_transactions.csv"

In [ ]:
df_raw_txns = spark.read.csv(txns_file, header=True)

In [ ]:
df_raw_txns.printSchema()
df_raw_txns.show(3, False)

In [ ]:
df_txns = (
    df_raw_txns.withColumnRenamed("CUST_ID", "cust_id")
    .withColumnRenamed("START_DATE", "start_date")
    .withColumnRenamed("END_DATE", "end_date")
    .withColumnRenamed("TRANS_ID", "txn_id")
    .withColumnRenamed("DATE", "date")
    .withColumnRenamed("YEAR", "year")
    .withColumnRenamed("MONTH", "month")
    .withColumnRenamed("DAY", "day")
    .withColumnRenamed("EXP_TYPE", "expense_type")
    .withColumnRenamed("AMOUNT", "amt")
)

In [ ]:
df_txns.printSchema()
df_txns.show(3, False)

# Setting Up Customer Data

In [ ]:
df_customer_det = spark.read.csv(customers_file, header=True)

In [ ]:
df_customer_det.printSchema()
df_customer_det.show(3, False)

In [ ]:
df_top5k_customers = (
    df_txns
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("distinct_txns"))
    .orderBy(F.desc("distinct_txns"))
    .limit(5000)
    .withColumn("row_id", F.row_number().over(Window.orderBy("cust_id")))
)

In [ ]:
df_customer_det = df_customer_det.withColumn("row_id", F.row_number().over(Window.orderBy("name")))
df_customer_identity = df_top5k_customers.join(df_customer_det, "row_id").drop("row_id")

In [ ]:
df_customer_identity.printSchema()
df_customer_identity.show(5, False)
df_customer_identity.select("cust_id").distinct().count()
df_customer_identity.count()

In [ ]:
def assign_city():
    cities = [
        'san_francisco', 'new_york', 'chicago', 'philadelphia',
        'boston', 'seattle', 'san_diego', 'los_angeles',
        'denver', 'portland'
    ]
    return random.choice(cities)

assign_city_udf = udf(assign_city, StringType())

In [ ]:
df_customer_identity = (
    df_customer_identity.withColumn(
        "city",
        assign_city_udf()
    )
)

In [ ]:
df_customer_identity.show(10, False)

# Write Customer Data

In [ ]:
(
    df_customer_identity
    .drop("distinct_txns")
    .write
    .mode("overwrite")
    .parquet("../../data/data_skew/customers.parquet")
)

# Write Skewed Transaction Data

In [ ]:
# df_top10k_customers.filter(F.col("distinct_txns") >= 7000).distinct().count()

In [ ]:
df_transactions = df_txns.join(
    df_top5k_customers,
    on="cust_id",
    how="inner"
).withColumn(
    "cust_id",
    F.when(
        F.col("distinct_txns") >= 8000, F.lit("C0YDPQWPBJ")
    ).otherwise(F.col("cust_id"))
)

In [ ]:
df_transactions = (
    df_transactions.withColumn(
        "city",
        assign_city_udf()
    )
)

In [ ]:
df_transactions.groupBy("cust_id").count().orderBy(F.desc("count")).show(5, False)
df_transactions.cache()

In [ ]:
df_transactions.printSchema()

In [ ]:
df_transactions.groupBy("cust_id", "city").count().orderBy(F.desc("count")).show(20, False)

In [ ]:
# Checks to validate if data is sane

# df_transactions.select("cust_id").distinct().count()
# df_transactions.select("txn_id").distinct().count()
# df_transactions.count()

In [ ]:
(
    df_transactions
    .drop("distinct_txns", "row_id")
    .write
    .mode("overwrite")
    .parquet("../../data/data_skew/transactions.parquet")
)

In [ ]:
# df_transactions_test = spark.read.parquet("../../data/data_skew/transactions.parquet")

In [ ]:
# Checks to validate if data is sane

# (
#     df_transactions_test
#     .groupBy("cust_id")
#     .agg(F.countDistinct("txn_id").alias("ct"))
#     .orderBy(F.desc("ct"))
#     .show(20, False)
# )

In [ ]:
spark.stop()